# Mackey–Glass: surrogate-only periodic-orbit construction

Purpose: discover and refine a periodic candidate using the fitted ODE, then
export a candidate for independent computer-assisted validation. The baseline
regression and forecasts match `mg_delay_forecasting.ipynb`.

Enable `RUN_PERIODIC_ORBIT_SEARCH` for the expensive construction workflow.
It uses a cubic discovery fit and a degree-five refinement fit with ridge
penalty `1e-7`, based on Euler reference data generated at internal step
`0.001` and sampled at `0.1`. Refinement uses the learned ODE on a scalar
Fourier delay manifold; there are no true-DDE Newton corrections before CAP.

New experiment files are written to `outputs/mg_surrogate_orbit_construction/`.
The final verification section inspects the existing frozen fixed-parameter
certificate. That certificate applies to its stored candidate, not automatically
to a new candidate generated here. `mg_periodic_orbit_certification.ipynb`
provides the focused theorem and certificate checks. MATLAB/INTLAB proof
regeneration is separately opt-in with `MG_RUN_CAP_FROM_NOTEBOOK=1` and uses
the documented frozen proof source.


## Environment and execution

From the repository root, install `python -m pip install -e ".[notebooks]"`.
Run this notebook from the root or its own directory. Shared code lives in `dpsr/`.
A default run fits and evaluates the baseline. Sweeps, long diagnostics, pruning,
and orbit searches are opt-in in the configuration cell. Generated results go
under `outputs/`; stored proof inputs remain separate in the CAP directories.


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "dpsr/polynomial.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from dpsr.polynomial import (
    PolynomialLibrary, PolynomialModel, LibraryModel, polynomial_exponents,
    polynomial_feature_names, fit_polynomial_vector_field, solve_dense_regression,
)
from dpsr.dynamics import model_rhs, rk4_step, cumulative_error_horizon, nrmse

from itertools import combinations_with_replacement, product
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.linalg import eig
from scipy.optimize import fsolve, root
from scipy.signal import find_peaks, welch
from scipy.spatial import cKDTree
from scipy.stats import wasserstein_distance
from scipy.io import savemat
import subprocess
import os

# Reproducibility and output policy
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
_root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    path for path in _root_candidates if (path / "CAP/Mackey-Glass_PO").exists()
)
NOTEBOOK_DIR = REPO_ROOT / "notebooks/mackey_glass"
OUTPUT_DIR = REPO_ROOT / "outputs/mg_surrogate_orbit_construction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional sweeps and long diagnostics are disabled for the baseline run.
RUN_EMBEDDING_DIAGNOSTICS = False
RUN_MODEL_SWEEP = False
RUN_CHAOS_DIAGNOSTICS = False
RUN_ADVANCED_DYNAMICS = False
RUN_PRUNING_SWEEPS = False
RUN_LIBRARY_COMPARISON = False
RUN_PRUNE_ADD_SWEEP = False
RUN_NOISE_SWEEP = False
RUN_PERIODIC_ORBIT_SEARCH = False

np.seterr(divide="ignore", invalid="ignore", over="ignore")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


## 1. Numerical configuration

The scalar Mackey-Glass delay differential equation is

\[
\dot{x}(t) =
\frac{\beta x(t-\tau)}{1+x(t-\tau)^n}-\gamma x(t).
\]

The delay-coordinate state is
\(X(t)=[x(t),x(t-\Delta),\ldots,x(t-(d-1)\Delta)]\).


In [ ]:
# Mackey-Glass parameters (unchanged from the source notebook)
BETA = 0.2
GAMMA = 0.1
NONLINEARITY = 10
DDE_DELAY = 17.0
HISTORY_VALUE = 1.2

# Time discretization and genuinely disjoint data windows
DT = 0.1
T_MAX = 5000.0
TRAIN_START = 117.0
TRAIN_END = 417.0
VALIDATION_START = 700.0
VALIDATION_END = 1300.0
TEST_START = 1500.0
TEST_END = 2500.0
ATTRACTOR_REFERENCE_START = 3500.0
ATTRACTOR_REFERENCE_END = 4500.0

# Chaos-aware selected model (chosen by the sweep below)
EMBEDDING_DELAY = 4.5
EMBEDDING_DIMENSION = 6
BASELINE_POLY_DEGREE = 3
BASELINE_RIDGE_ALPHA = 1e-4
RIDGE_DEGREE_WEIGHT = 1.0

# Shared rollout metrics
HORIZON_CAP = 150.0
CUMULATIVE_ERROR_TOLERANCE = 0.05
STATE_BOUNDS = (0.15, 1.65)

TOTAL_STEPS = int(round(T_MAX / DT))
TAU_STEPS = int(round(DDE_DELAY / DT))
TRAIN_START_STEPS = int(round(TRAIN_START / DT))
TRAIN_END_STEPS = int(round(TRAIN_END / DT))
VALIDATION_START_STEPS = int(round(VALIDATION_START / DT))
VALIDATION_END_STEPS = int(round(VALIDATION_END / DT))
TEST_START_STEPS = int(round(TEST_START / DT))
TEST_END_STEPS = int(round(TEST_END / DT))


## 2. Shared utilities

These functions replace the repeated implementations that appeared throughout the
original notebook. Every experiment below uses the same delay-embedding, RK4 rollout,
predictive-horizon, NRMSE, and model-evaluation definitions.


In [ ]:
from dpsr.systems import generate_mackey_glass_euler, build_delay_embedding


def fit_delay_polynomial_model(
    X, dt, degree=3, alpha=BASELINE_RIDGE_ALPHA,
    degree_weight=RIDGE_DEGREE_WEIGHT,
):
    """Dense normalized ridge; derivative estimation stays specific to MG."""
    return fit_polynomial_vector_field(
        X, np.gradient(X, dt, axis=0, edge_order=2), degree=degree,
        regularization=alpha, normalize=True, degree_weight=degree_weight,
    )


def safe_rk4_rollout(model, x0, times, bounds=STATE_BOUNDS):
    times = np.asarray(times)
    if len(times) < 2:
        raise ValueError("At least two time points are required.")
    dt = float(times[1] - times[0])
    lower, upper = bounds
    trajectory = np.zeros((len(times), len(x0)), dtype=float)
    trajectory[0] = x0
    for i in range(1, len(times)):
        try:
            next_state = rk4_step(model, trajectory[i - 1], dt)
        except Exception:
            return None
        if not np.all(np.isfinite(next_state)) or np.any(next_state < lower) or np.any(next_state > upper):
            return None
        trajectory[i] = next_state
    return trajectory


def predictive_horizon(X_true, X_pred, dt, tolerance=CUMULATIVE_ERROR_TOLERANCE):
    return cumulative_error_horizon(X_true, X_pred, dt, tolerance, coordinate=0)


def normalized_rmse(X_true, X_pred):
    return nrmse(X_true, X_pred, coordinate=0)


def evenly_spaced_test_indices(n_samples, horizon_length, n_initial_conditions=20):
    latest = n_samples - horizon_length
    if latest <= 0:
        raise ValueError("The rollout horizon is longer than the available data.")
    return np.linspace(0, latest - 1, min(n_initial_conditions, latest), dtype=int)


def evaluate_model(model, X_reference, times, test_indices, tolerance=CUMULATIVE_ERROR_TOLERANCE, bounds=STATE_BOUNDS):
    horizons, errors = [], []
    stable_rollouts = 0
    for idx in test_indices:
        truth = X_reference[idx : idx + len(times)]
        prediction = safe_rk4_rollout(model, truth[0], times, bounds=bounds)
        if prediction is None:
            continue
        stable_rollouts += 1
        horizons.append(predictive_horizon(truth, prediction, times[1] - times[0], tolerance))
        errors.append(normalized_rmse(truth, prediction))
    return {
        "stable_rollouts": stable_rollouts,
        "stable_fraction": stable_rollouts / len(test_indices),
        "mean_horizon": float(np.mean(horizons)) if horizons else np.nan,
        "median_horizon": float(np.median(horizons)) if horizons else np.nan,
        "mean_nrmse": float(np.mean(errors)) if errors else np.nan,
    }


def spectral_entropy(signal, dt=DT):
    _, power = welch(signal - np.mean(signal), fs=1.0 / dt, nperseg=min(2048, len(signal)))
    power = power[1:]
    power = power / (power.sum() + 1e-30)
    return float(-np.sum(power * np.log(power + 1e-30)) / np.log(len(power)))


def autocorrelation_vector(signal, max_lag_time=100.0, stride=10):
    centered = np.asarray(signal) - np.mean(signal)
    spectrum = np.fft.rfft(centered, n=2 * len(centered))
    acf = np.fft.irfft(spectrum * np.conj(spectrum))[: len(centered)]
    acf /= acf[0] + 1e-30
    stop = min(len(acf), int(round(max_lag_time / DT)) + 1)
    return acf[:stop:stride]


def compare_attractor_statistics(predicted, reference):
    predicted = np.asarray(predicted)
    reference = np.asarray(reference)
    pred_acf = autocorrelation_vector(predicted)
    ref_acf = autocorrelation_vector(reference)
    n = min(len(pred_acf), len(ref_acf))
    return {
        "std_ratio": float(np.std(predicted) / (np.std(reference) + 1e-12)),
        "mean_error_std_units": float(abs(np.mean(predicted) - np.mean(reference)) / (np.std(reference) + 1e-12)),
        "wasserstein_std_units": float(wasserstein_distance(predicted, reference) / (np.std(reference) + 1e-12)),
        "spectral_entropy": spectral_entropy(predicted),
        "reference_spectral_entropy": spectral_entropy(reference),
        "acf_rmse": float(np.sqrt(np.mean((pred_acf[:n] - ref_acf[:n]) ** 2))),
    }


def save_figure(fig, filename, dpi=200):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


## 3. Generate the Mackey-Glass trajectory and delay embedding

The target coordinate begins one DDE-delay after `TRAIN_START`. This preserves the
alignment convention used by the original notebook while making the effective time
labels explicit. With the default settings, the four coordinate histories begin near
the requested training boundary.


In [ ]:
time_full, data = generate_mackey_glass_euler(
    beta=BETA, gamma=GAMMA, exponent=NONLINEARITY, delay=DDE_DELAY,
    dt=DT, t_max=T_MAX, history_value=HISTORY_VALUE,
)

delay_steps = int(round(EMBEDDING_DELAY / DT))
X_delayed = build_delay_embedding(
    data, TRAIN_START_STEPS, TRAIN_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
X_validation = build_delay_embedding(
    data, VALIDATION_START_STEPS, VALIDATION_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
X_test = build_delay_embedding(
    data, TEST_START_STEPS, TEST_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
t_delayed = np.arange(TRAIN_START_STEPS, TRAIN_END_STEPS) * DT
x_t = X_delayed[:, 0]
attractor_reference = data[
    int(round(ATTRACTOR_REFERENCE_START / DT)) : int(round(ATTRACTOR_REFERENCE_END / DT)) + 1
]

print(f"Scalar trajectory shape: {data.shape}")
print(f"Training embedding:   {X_delayed.shape}, t={TRAIN_START:.1f}..{TRAIN_END:.1f}")
print(f"Validation embedding: {X_validation.shape}, t={VALIDATION_START:.1f}..{VALIDATION_END:.1f}")
print(f"Test embedding:       {X_test.shape}, t={TEST_START:.1f}..{TEST_END:.1f}")


## 4. Optional embedding diagnostics

Average mutual information (AMI) suggests a delay, while false nearest neighbors (FNN)
assesses whether the embedding dimension is large enough. These computations are
exploratory and are disabled by default.


In [ ]:
def mutual_information_histogram(x, y, bins=20):
    joint, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = joint / joint.sum()
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    expected = px[:, None] * py[None, :]
    valid = (pxy > 0) & (expected > 0)
    return float(np.sum(pxy[valid] * np.log(pxy[valid] / expected[valid])))


def average_mutual_information(series, lag_steps, bins=20):
    scores = []
    for lag in lag_steps:
        scores.append(
            mutual_information_histogram(series[:-lag], series[lag:], bins=bins)
        )
    return np.asarray(scores)


def first_local_minimum(values):
    for i in range(1, len(values) - 1):
        if values[i] < values[i - 1] and values[i] < values[i + 1]:
            return i
    return int(np.argmin(values))


def scalar_delay_embedding(series, delay_steps, dimension):
    n = len(series) - dimension * delay_steps
    if n <= 1:
        raise ValueError("Series is too short for the requested embedding.")
    return np.column_stack(
        [series[j * delay_steps : j * delay_steps + n] for j in range(dimension)]
    )


def false_nearest_neighbor_fractions(
    series,
    delay_steps,
    max_dimension=8,
    ratio_threshold=15.0,
    sample_size=1000,
    seed=RANDOM_SEED,
):
    """Estimate the FNN fraction using nearest neighbors in successive embeddings."""
    local_rng = np.random.default_rng(seed)
    full = scalar_delay_embedding(series, delay_steps, max_dimension + 1)
    fractions = []

    for dimension in range(1, max_dimension + 1):
        current = full[:, :dimension]
        expanded = full[:, : dimension + 1]
        sample = local_rng.choice(
            len(current), size=min(sample_size, len(current)), replace=False
        )
        tree = cKDTree(current)
        distances, neighbors = tree.query(current[sample], k=2)
        nearest_distance = distances[:, 1]
        nearest_index = neighbors[:, 1]
        added_distance = np.abs(
            expanded[sample, dimension] - expanded[nearest_index, dimension]
        )
        ratio = added_distance / np.maximum(nearest_distance, 1e-12)
        fractions.append(float(np.mean(ratio > ratio_threshold)))
    return np.asarray(fractions)


if RUN_EMBEDDING_DIAGNOSTICS:
    lag_steps = np.arange(1, 400)
    ami = average_mutual_information(data, lag_steps)
    best_lag_index = first_local_minimum(ami)
    selected_delay_steps = int(lag_steps[best_lag_index])

    fnn = false_nearest_neighbor_fractions(data, selected_delay_steps)
    dimensions = np.arange(1, len(fnn) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(lag_steps * DT, ami, color="steelblue")
    axes[0].axvline(selected_delay_steps * DT, color="crimson", ls="--")
    axes[0].set(xlabel="Lag (t.u.)", ylabel="Average mutual information", title="AMI")

    axes[1].plot(dimensions, 100 * fnn, "o-", color="darkgreen")
    axes[1].axhline(5, color="crimson", ls="--", label="5% threshold")
    axes[1].set(
        xlabel="Embedding dimension",
        ylabel="False nearest neighbors (%)",
        title="FNN diagnostic",
    )
    axes[1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_embedding_diagnostics.png")
    plt.show()

    print(f"First AMI minimum: {selected_delay_steps * DT:.2f} t.u.")
else:
    print("Embedding diagnostics skipped. Set RUN_EMBEDDING_DIAGNOSTICS = True to run.")


## 5. Chaos-aware selected polynomial model

The model below uses the selected hyperparameters and normalized ridge regression. Unlike the source notebook, all reported predictive metrics use later, unseen windows.


In [ ]:
fitted_polynomial = fit_delay_polynomial_model(
    X_delayed,
    dt=DT,
    degree=BASELINE_POLY_DEGREE,
    alpha=BASELINE_RIDGE_ALPHA,
)

print("Learned chaos-aware model:")
fitted_polynomial.print()
print(f"\nCoefficient matrix shape: {fitted_polynomial.coefficients().shape}")
print(
    f"Selected parameters: delay={EMBEDDING_DELAY}, dimension={EMBEDDING_DIMENSION}, "
    f"degree={BASELINE_POLY_DEGREE}, alpha={BASELINE_RIDGE_ALPHA:g}"
)
model = fitted_polynomial


In [ ]:
baseline_times = np.arange(len(X_delayed)) * DT
X_sim_delayed = safe_rk4_rollout(model, X_delayed[0], baseline_times)

if X_sim_delayed is None:
    warnings.warn("The selected rollout left the configured safety bounds.")
else:
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(X_delayed[:, 0], X_delayed[:, 1], X_delayed[:, 2], "r--", lw=0.6, alpha=0.7, label="Mackey-Glass data")
    ax1.plot(X_sim_delayed[:, 0], X_sim_delayed[:, 1], X_sim_delayed[:, 2], "k-", lw=0.7, alpha=0.8, label="learned rollout")
    ax1.set(xlabel="$x(t)$", ylabel=f"$x(t-{EMBEDDING_DELAY:g})$", zlabel=f"$x(t-{2 * EMBEDDING_DELAY:g})$", title="Training attractor")
    ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(122)
    ax2.plot(t_delayed, X_delayed[:, 0], "r--", lw=0.8, label="Mackey-Glass data")
    ax2.plot(t_delayed, X_sim_delayed[:, 0], "k-", lw=0.9, label="learned rollout")
    ax2.set(xlabel="Time", ylabel="$x(t)$", title="Training-window rollout")
    ax2.legend(fontsize=8)
    plt.tight_layout()
    save_figure(fig, "mg_chaos_selected_training_fit.png")
    plt.show()


In [ ]:
horizon_times = np.arange(0.0, HORIZON_CAP + DT, DT)
validation_indices = evenly_spaced_test_indices(
    len(X_validation), len(horizon_times), n_initial_conditions=12
)
validation_metrics = evaluate_model(model, X_validation, horizon_times, validation_indices)
print(pd.Series(validation_metrics, name="held-out validation"))

representative_idx = int(validation_indices[len(validation_indices) // 2])
X_true_example = X_validation[representative_idx : representative_idx + len(horizon_times)]
X_pred_example = safe_rk4_rollout(model, X_true_example[0], horizon_times)

if X_pred_example is not None:
    cumulative_error = np.cumsum((X_true_example[:, 0] - X_pred_example[:, 0]) ** 2) * DT
    horizon = predictive_horizon(X_true_example, X_pred_example, DT)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(horizon_times, X_true_example[:, 0], "r--", lw=1, label="unseen truth")
    axes[0].plot(horizon_times, X_pred_example[:, 0], "k-", lw=1, label="prediction")
    axes[0].set(xlabel="Forecast time", ylabel="$x(t)$", title="Representative held-out forecast")
    axes[0].legend()
    axes[1].plot(horizon_times, cumulative_error, color="darkgreen", lw=2)
    axes[1].axhline(CUMULATIVE_ERROR_TOLERANCE, color="crimson", ls="--", label="tolerance")
    axes[1].axvline(horizon, color="darkorange", ls=":", label=f"horizon = {horizon:.1f}")
    axes[1].set(xlabel="Forecast time", ylabel="Cumulative squared error", title="Predictive-horizon calculation")
    axes[1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_chaos_heldout_forecast.png")
    plt.show()


## 6. Chaos-aware hyperparameter sweep

The sweep uses the training window only for fitting. It ranks models on a separate validation window using short-term NRMSE plus long-run Wasserstein distance, autocorrelation mismatch, spectral-entropy mismatch, and amplitude mismatch. The broad development sweep covered delays 4–7, dimensions 3–6, degrees 2–3, and eight ridge strengths; the compact reproducible sweep below covers the neighborhood of the winner.


In [ ]:
def run_polynomial_model_sweep(
    series,
    delays=(4.0, 4.5, 5.0, 5.5),
    degrees=(3,),
    dimensions=(4, 5, 6),
    alphas=(1e-6, 1e-4, 1e-3, 1e-2),
    n_initial_conditions=3,
):
    rows = []
    combinations = list(product(delays, degrees, dimensions, alphas))
    short_times = np.arange(0.0, 100.0 + DT, DT)
    # A 1,000-unit rollout is long enough to expose the limit-cycle collapse
    # that shorter health checks can miss.
    long_times = np.arange(0.0, 1000.0 + DT, DT)

    for run_number, (delay, degree, dimension, alpha) in enumerate(combinations, start=1):
        started = time.perf_counter()
        lag = int(round(delay / DT))
        try:
            X_train_candidate = build_delay_embedding(series, TRAIN_START_STEPS, TRAIN_END_STEPS, lag, dimension)
            X_validation_candidate = build_delay_embedding(series, VALIDATION_START_STEPS, VALIDATION_END_STEPS, lag, dimension)
            candidate = fit_delay_polynomial_model(X_train_candidate, DT, degree=degree, alpha=alpha)
            candidate_fast = candidate
            indices = evenly_spaced_test_indices(len(X_validation_candidate), len(short_times), n_initial_conditions)
            metrics = evaluate_model(candidate_fast, X_validation_candidate, short_times, indices)
            long_trajectory = safe_rk4_rollout(candidate_fast, X_validation_candidate[500], long_times)
            if long_trajectory is None:
                attractor = {"std_ratio": np.nan, "wasserstein_std_units": np.nan, "spectral_entropy": np.nan, "reference_spectral_entropy": np.nan, "acf_rmse": np.nan}
                score = np.inf
            else:
                attractor = compare_attractor_statistics(long_trajectory[2000:, 0], attractor_reference)
                entropy_error = abs(attractor["spectral_entropy"] - attractor["reference_spectral_entropy"])
                if metrics["stable_fraction"] == 1.0:
                    score = (
                        metrics["mean_nrmse"]
                        + 1.5 * attractor["wasserstein_std_units"]
                        + 1.5 * attractor["acf_rmse"]
                        + 3.0 * entropy_error
                        + 0.5 * abs(np.log(max(attractor["std_ratio"], 1e-8)))
                    )
                else:
                    score = np.inf
            rows.append({
                "delay": delay, "degree": degree, "dimension": dimension, "alpha": alpha,
                **metrics, **attractor, "selection_score": score,
                "elapsed_seconds": time.perf_counter() - started,
            })
        except Exception as exc:
            rows.append({"delay": delay, "degree": degree, "dimension": dimension, "alpha": alpha, "selection_score": np.inf, "error": str(exc)})

    return pd.DataFrame(rows).sort_values("selection_score", ascending=True, ignore_index=True)


if RUN_MODEL_SWEEP:
    df_model_sweep = run_polynomial_model_sweep(data)
    df_model_sweep.to_csv(OUTPUT_DIR / "mg_chaos_aware_model_sweep.csv", index=False)
    display(df_model_sweep.head(12))
    print("Configured winner:", {"delay": EMBEDDING_DELAY, "dimension": EMBEDDING_DIMENSION, "degree": BASELINE_POLY_DEGREE, "alpha": BASELINE_RIDGE_ALPHA})
else:
    print("Chaos-aware sweep skipped. Set RUN_MODEL_SWEEP = True to run.")


## 7. Long-run chaos and truly held-out test diagnostics

This section directly checks the failure mode described in the request. The selected model is compared with the original `(delay=6, dimension=4, degree=3, alpha=0)` fit on a later test window. Positive finite-time divergence alone is not accepted: the surrogate must remain bounded, preserve attractor statistics, and produce a positive maximal Lyapunov estimate across multiple initial conditions.


In [ ]:
def estimate_maximal_lyapunov(
    model,
    x0,
    dt,
    total_time=600.0,
    transient_time=200.0,
    renormalization_time=1.0,
    perturbation=1e-7,
    bounds=STATE_BOUNDS,
    seed=RANDOM_SEED,
):
    """Benettin estimate with repeated renormalization and a long burn-in."""
    local_rng = np.random.default_rng(seed)
    lower, upper = bounds
    reference = np.asarray(x0, dtype=float).copy()
    for _ in range(int(round(transient_time / dt))):
        reference = rk4_step(model, reference, dt)
        if not np.all(np.isfinite(reference)) or np.any(reference < lower) or np.any(reference > upper):
            return np.nan, {"status": "failed during transient"}

    direction = local_rng.normal(size=len(reference))
    direction /= np.linalg.norm(direction)
    perturbed = reference + perturbation * direction
    block_steps = max(1, int(round(renormalization_time / dt)))
    block_time = block_steps * dt
    n_blocks = int(round(total_time / block_time))
    cumulative_growth = 0.0
    times, running = [], []

    for block in range(n_blocks):
        for _ in range(block_steps):
            reference = rk4_step(model, reference, dt)
            perturbed = rk4_step(model, perturbed, dt)
            if (
                not np.all(np.isfinite(reference)) or not np.all(np.isfinite(perturbed))
                or np.any(reference < lower) or np.any(reference > upper)
                or np.any(perturbed < lower) or np.any(perturbed > upper)
            ):
                return np.nan, {"status": "failed during rollout", "times": np.asarray(times), "running_mle": np.asarray(running)}
        delta = perturbed - reference
        delta_norm = np.linalg.norm(delta)
        if not np.isfinite(delta_norm) or delta_norm <= 0:
            return np.nan, {"status": "invalid perturbation"}
        cumulative_growth += np.log(delta_norm / perturbation)
        elapsed = (block + 1) * block_time
        times.append(elapsed)
        running.append(cumulative_growth / elapsed)
        perturbed = reference + perturbation * delta / delta_norm

    return float(cumulative_growth / (n_blocks * block_time)), {
        "status": "completed", "times": np.asarray(times), "running_mle": np.asarray(running)
    }


def stress_test_model(label, fitted_model, X_reference, long_reference):
    fast_model = fitted_model
    test_times = np.arange(0.0, HORIZON_CAP + DT, DT)
    test_indices = evenly_spaced_test_indices(len(X_reference), len(test_times), 12)
    forecast = evaluate_model(fast_model, X_reference, test_times, test_indices)

    long_times = np.arange(0.0, 2000.0 + DT, DT)
    attractor_rows, mle_values = [], []
    for j, index in enumerate([0, 1000, 3000]):
        long_rollout = safe_rk4_rollout(fast_model, X_reference[index], long_times)
        if long_rollout is not None:
            attractor_rows.append(compare_attractor_statistics(long_rollout[5000:, 0], long_reference))
        for seed in [11 + j, 31 + j]:
            mle, _ = estimate_maximal_lyapunov(fast_model, X_reference[index], DT, seed=seed)
            if np.isfinite(mle):
                mle_values.append(mle)

    return {
        "model": label,
        **forecast,
        "stable_long_rollouts": len(attractor_rows),
        "mean_wasserstein": float(np.mean([r["wasserstein_std_units"] for r in attractor_rows])) if attractor_rows else np.nan,
        "mean_std_ratio": float(np.mean([r["std_ratio"] for r in attractor_rows])) if attractor_rows else np.nan,
        "mean_spectral_entropy": float(np.mean([r["spectral_entropy"] for r in attractor_rows])) if attractor_rows else np.nan,
        "reference_spectral_entropy": spectral_entropy(long_reference),
        "mle_mean": float(np.mean(mle_values)) if mle_values else np.nan,
        "mle_std": float(np.std(mle_values)) if mle_values else np.nan,
        "mle_successes": len(mle_values),
    }


if RUN_CHAOS_DIAGNOSTICS:
    original_X_train = build_delay_embedding(data, TRAIN_START_STEPS, TRAIN_END_STEPS, int(round(6.0 / DT)), 4)
    original_X_test = build_delay_embedding(data, TEST_START_STEPS, TEST_END_STEPS, int(round(6.0 / DT)), 4)
    original_fit = fit_delay_polynomial_model(original_X_train, DT, degree=3, alpha=0.0)

    comparison = pd.DataFrame([
        stress_test_model("original baseline", original_fit, original_X_test, attractor_reference),
        stress_test_model("chaos-aware selected",fitted_polynomial, X_test, attractor_reference),
    ]).set_index("model")
    display(comparison.T)
    comparison.to_csv(OUTPUT_DIR / "mg_chaos_stress_test.csv")

    mle, mle_diagnostics = estimate_maximal_lyapunov(model, X_test[1000], DT, seed=31)
    long_times = np.arange(0.0, 2000.0 + DT, DT)
    long_selected = safe_rk4_rollout(model, X_test[1000], long_times)
    if long_selected is None:
        raise RuntimeError("Selected model failed its long-run stability check.")
    long_stats = compare_attractor_statistics(long_selected[5000:, 0], attractor_reference)
    print(f"Selected-model MLE: {mle:.6f} ({mle_diagnostics['status']})")
    print("Selected long-run statistics:", long_stats)
    if not (np.isfinite(mle) and mle > 0.001):
        warnings.warn("The final MLE did not pass the positive-chaos threshold.")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(long_times, long_selected[:, 0], color="black", lw=0.45)
    axes[0, 0].set(xlabel="Time", ylabel="$x(t)$", title="2,000-unit learned rollout (bounded, non-collapsing)")
    axes[0, 1].plot(long_selected[:, 0], long_selected[:, 1], color="black", lw=0.35, alpha=0.7)
    axes[0, 1].set(xlabel="$x(t)$", ylabel=f"$x(t-{EMBEDDING_DELAY:g})$", title="Learned long-run attractor")
    axes[1, 0].hist(attractor_reference, bins=60, density=True, alpha=0.55, color="crimson", label="true")
    axes[1, 0].hist(long_selected[5000:, 0], bins=60, density=True, alpha=0.55, color="black", label="learned")
    axes[1, 0].set(xlabel="$x$", ylabel="Density", title="Invariant-distribution comparison")
    axes[1, 0].legend()
    axes[1, 1].plot(mle_diagnostics["times"], mle_diagnostics["running_mle"], color="darkgreen")
    axes[1, 1].axhline(mle, color="crimson", ls="--", label=f"final MLE = {mle:.4f}")
    axes[1, 1].axhline(0.0, color="black", lw=0.8)
    axes[1, 1].set(xlabel="Accumulation time", ylabel="Running MLE", title="Benettin MLE convergence")
    axes[1, 1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_chaos_long_run_diagnostics.png")
    plt.show()
else:
    print("Chaos diagnostics skipped. Set RUN_CHAOS_DIAGNOSTICS = True to run.")


### 7.1 Very-long-time limit-cycle exclusion test

The preceding 2,000-unit test is extended here to **20,000 time units** from three test-set initial conditions. Only the final 5,000 units are used for recurrence diagnostics. A stable limit cycle would produce near-perfect recurrence at one lag, a small set of repeated peak heights, and a largest Lyapunov exponent near zero after a long burn-in. The selected model is required to show the opposite behavior while retaining stationary variance and spectral complexity.


In [ ]:
def limit_cycle_diagnostics(signal, reference):
    signal = np.asarray(signal)
    centered = signal - np.mean(signal)
    spectrum = np.fft.rfft(centered, n=2 * len(centered))
    acf = np.fft.irfft(spectrum * np.conj(spectrum))[: len(centered)]
    acf /= acf[0] + 1e-30
    lag_start = int(round(10.0 / DT))
    lag_stop = min(len(acf), int(round(200.0 / DT)) + 1)
    best_lag = lag_start + int(np.argmax(acf[lag_start:lag_stop]))
    recurrence_nrmse = float(
        np.sqrt(np.mean((signal[best_lag:] - signal[:-best_lag]) ** 2))
        / (np.std(signal) + 1e-30)
    )
    peaks, _ = find_peaks(signal, distance=int(round(2.0 / DT)), prominence=0.005)
    peak_values = signal[peaks]
    return {
        "std_ratio_to_true": float(np.std(signal) / np.std(reference)),
        "spectral_entropy": spectral_entropy(signal),
        "best_recurrence_lag": float(best_lag * DT),
        "best_recurrence_acf": float(acf[best_lag]),
        "best_recurrence_nrmse": recurrence_nrmse,
        "late_peak_count": int(len(peak_values)),
        "unique_peak_heights_1e-4": int(len(np.unique(np.round(peak_values, 4)))),
    }


if RUN_CHAOS_DIAGNOSTICS:
    very_long_times = np.arange(0.0, 20000.0 + DT, DT)
    very_long_rows = []
    for run_id, test_index in enumerate([0, 1000, 3000], start=1):
        trajectory = safe_rk4_rollout(model, X_test[test_index], very_long_times)
        if trajectory is None:
            very_long_rows.append({"run": run_id, "bounded": False})
            continue
    
        final_window = trajectory[-int(round(5000.0 / DT)) :, 0]
        diagnostics = limit_cycle_diagnostics(final_window, attractor_reference)
        window_stds, window_entropies = [], []
        window_steps = int(round(5000.0 / DT))
        for window_id in range(4):
            window = trajectory[window_id * window_steps : (window_id + 1) * window_steps, 0]
            window_stds.append(float(np.std(window)))
            window_entropies.append(spectral_entropy(window))
    
        post_burn_mle, _ = estimate_maximal_lyapunov(
            model,
            X_test[test_index],
            DT,
            transient_time=5000.0,
            total_time=2000.0,
            renormalization_time=1.0,
            seed=100 + run_id,
        )
        very_long_rows.append({
            "run": run_id,
            "bounded": True,
            "duration": 20000.0,
            "post_5000_mle": post_burn_mle,
            "window_std_minmax_ratio": min(window_stds) / max(window_stds),
            "window_entropy_min": min(window_entropies),
            **diagnostics,
        })
    
    very_long_results = pd.DataFrame(very_long_rows).set_index("run")
    display(very_long_results.T)
    very_long_results.to_csv(OUTPUT_DIR / "mg_very_long_limit_cycle_test.csv")
    
    limit_cycle_excluded = bool(
        very_long_results["bounded"].all()
        and (very_long_results["post_5000_mle"] > 0.001).all()
        and (very_long_results["window_std_minmax_ratio"] > 0.95).all()
        and (very_long_results["best_recurrence_nrmse"] > 0.20).all()
        and (very_long_results["unique_peak_heights_1e-4"] > 50).all()
    )
    print("Very-long-time limit-cycle exclusion:", "PASS" if limit_cycle_excluded else "FAIL")
    if not limit_cycle_excluded:
        warnings.warn("One or more very-long-time chaos criteria failed.")
else:
    print("Very-long-time diagnostics skipped. Set RUN_CHAOS_DIAGNOSTICS = True to run.")


## 8. Advanced dynamics analysis

This optional section collects the original notebook's new-history, equilibrium,
eigenspace, manifold, and DDE-spectrum experiments. It is disabled by default because
it is exploratory and relies on additional numerical assumptions.


In [ ]:
def fit_polynomial_history(
    learned_trajectory,
    delay=DDE_DELAY,
    dt=DT,
    minimum_degree=3,
    maximum_degree=29,
    error_tolerance=1e-5,
):
    """Fit a polynomial to one delay-length window of a learned trajectory."""
    n_window = int(round(delay / dt))
    window_time = np.arange(n_window) * dt
    window_values = np.asarray(learned_trajectory)[:n_window, 0]

    best_polynomial, best_degree, best_error = None, None, np.inf
    for degree in range(minimum_degree, maximum_degree + 1):
        coefficients = np.polyfit(window_time, window_values, degree)
        polynomial = np.poly1d(coefficients)
        error = float(np.max(np.abs(polynomial(window_time) - window_values)))
        best_polynomial, best_degree, best_error = polynomial, degree, error
        if error < error_tolerance:
            break
    return best_polynomial, best_degree, best_error


def make_history_function(polynomial, delay=DDE_DELAY):
    """Map t in [-delay, 0] to the fitted polynomial's [0, delay] interval."""
    def history(t):
        shifted = np.clip(t + delay, 0.0, delay)
        return float(polynomial(shifted))
    return history


def simulate_mg_with_history(
    history,
    times,
    beta=BETA,
    gamma=GAMMA,
    exponent=NONLINEARITY,
    delay=DDE_DELAY,
):
    """Integrate the true Mackey-Glass DDE with ddeint and a supplied history."""
    try:
        from ddeint import ddeint
    except ImportError as exc:
        raise ImportError(
            "This optional experiment requires `python -m pip install ddeint`."
        ) from exc

    def right_hand_side(Y, t):
        x_now = Y(t)
        x_delay = Y(t - delay)
        return beta * x_delay / (1.0 + x_delay**exponent) - gamma * x_now

    solution = ddeint(right_hand_side, history, np.asarray(times))
    return solution[:, 0] if solution.ndim > 1 else solution


if RUN_ADVANCED_DYNAMICS:
    if X_sim_delayed is None:
        raise RuntimeError("A bounded baseline rollout is required for the history experiment.")

    history_polynomial, history_degree, history_error = fit_polynomial_history(
        X_sim_delayed
    )
    history_function = make_history_function(history_polynomial)
    fresh_times = np.arange(0.0, 200.0, DT)
    true_from_fitted_history = simulate_mg_with_history(
        history_function, fresh_times
    )

    ideal_embedded_ic = np.array(
        [
            history_function(-index * EMBEDDING_DELAY)
            for index in range(EMBEDDING_DIMENSION)
        ]
    )
    nearest_index = int(
        np.argmin(np.linalg.norm(X_delayed - ideal_embedded_ic, axis=1))
    )
    safe_embedded_ic = X_delayed[nearest_index]
    learned_from_history = safe_rk4_rollout(
        model, safe_embedded_ic, fresh_times
    )

    print(
        f"History polynomial degree={history_degree}, "
        f"max fitting error={history_error:.3e}, "
        f"nearest embedded-data index={nearest_index}"
    )

    if learned_from_history is not None:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(
            fresh_times,
            true_from_fitted_history,
            "r--",
            lw=0.9,
            label="True MG with fitted history",
        )
        ax.plot(
            fresh_times,
            learned_from_history[:, 0],
            "k-",
            lw=0.9,
            label="dense polynomial regression rollout",
        )
        ax.set(
            xlabel="Time",
            ylabel="$x(t)$",
            title="New-history-function trajectory comparison",
        )
        ax.legend()
        save_figure(fig, "mg_new_history_comparison.png")
        plt.show()


In [ ]:
def find_equilibria(model, dimension, n_starts=2000, bounds=(0.0, 2.0), seed=RANDOM_SEED):
    """Find and deduplicate roots of the learned vector field."""
    local_rng = np.random.default_rng(seed)

    def field(z):
        return model_rhs(model, z)

    roots = []
    for guess in local_rng.uniform(bounds[0], bounds[1], size=(n_starts, dimension)):
        try:
            solution, info, status, _ = fsolve(field, guess, full_output=True)
            if status != 1 or np.linalg.norm(field(solution)) >= 1e-8:
                continue
            if all(np.linalg.norm(solution - existing) > 1e-4 for existing in roots):
                roots.append(solution)
        except Exception:
            continue
    return roots


def numerical_jacobian(model, point, epsilon=1e-6):
    """Centered finite-difference Jacobian of a learned vector field."""
    point = np.asarray(point)
    jacobian = np.zeros((len(point), len(point)))
    for j in range(len(point)):
        perturbation = np.zeros(len(point))
        perturbation[j] = epsilon
        jacobian[:, j] = (
            model_rhs(model, point + perturbation)
            - model_rhs(model, point - perturbation)
        ) / (2 * epsilon)
    return jacobian


def classify_equilibrium(eigenvalues, tolerance=1e-10):
    real_parts = np.real(eigenvalues)
    if np.all(real_parts < -tolerance):
        return "sink"
    if np.all(real_parts > tolerance):
        return "source"
    if np.any(real_parts < -tolerance) and np.any(real_parts > tolerance):
        return "saddle"
    return "non-hyperbolic"


def analyze_equilibria(model, roots, X_reference):
    records = []
    for point in roots:
        jacobian = numerical_jacobian(model, point)
        eigenvalues, eigenvectors = eig(jacobian)
        records.append(
            {
                "z_star": point,
                "jacobian": jacobian,
                "eigenvalues": eigenvalues,
                "eigenvectors": eigenvectors,
                "classification": classify_equilibrium(eigenvalues),
                "min_distance_to_attractor": float(
                    np.min(np.linalg.norm(X_reference - point, axis=1))
                ),
            }
        )
    return sorted(records, key=lambda record: record["min_distance_to_attractor"])


def dde_characteristic_roots(
    gamma,
    linearized_delayed_coefficient,
    delay,
    real_range=(-1.0, 0.2),
    imag_range=(-2.0, 2.0),
    grid=(20, 40),
):
    """Numerically solve lambda + gamma - a exp(-lambda tau) = 0."""
    def residual(z):
        lam = z[0] + 1j * z[1]
        value = lam + gamma - linearized_delayed_coefficient * np.exp(-lam * delay)
        return [value.real, value.imag]

    roots = []
    for real_guess in np.linspace(*real_range, grid[0]):
        for imag_guess in np.linspace(*imag_range, grid[1]):
            solution = root(residual, [real_guess, imag_guess])
            if not solution.success:
                continue
            lam = solution.x[0] + 1j * solution.x[1]
            if all(abs(lam - existing) >= 1e-5 for existing in roots):
                roots.append(lam)
    return sorted(roots, key=lambda value: value.real, reverse=True)


if RUN_ADVANCED_DYNAMICS:
    equilibrium_roots = find_equilibria(model, X_delayed.shape[1])
    equilibria = analyze_equilibria(model, equilibrium_roots, X_delayed)

    print(f"Found {len(equilibria)} distinct learned equilibria.")
    for i, record in enumerate(equilibria):
        print(
            f"{i}: z*={np.round(record['z_star'], 5)}, "
            f"class={record['classification']}, "
            f"distance={record['min_distance_to_attractor']:.4f}"
        )
        for value in sorted(record["eigenvalues"], key=lambda item: item.real):
            print(f"    {value.real:+.6f} {value.imag:+.6f}i")

    leading_dde_roots = dde_characteristic_roots(
        gamma=GAMMA,
        linearized_delayed_coefficient=-0.4,
        delay=DDE_DELAY,
    )
    print("\nLeading DDE characteristic roots:")
    for value in leading_dde_roots[:20]:
        print(value)

    if equilibria:
        selected_equilibrium = equilibria[0]
        fig = plt.figure(figsize=(13, 5.5))
        ax = fig.add_subplot(121, projection="3d")
        ax.plot(
            X_delayed[:, 0],
            X_delayed[:, 1],
            X_delayed[:, 2],
            "r--",
            lw=0.5,
            alpha=0.4,
            label="Attractor",
        )
        point = selected_equilibrium["z_star"]
        ax.scatter(*point[:3], s=140, color="black", label="Nearest equilibrium")
        ax.set(
            xlabel="$x(t)$",
            ylabel=f"$x(t-{EMBEDDING_DELAY:g})$",
            zlabel=f"$x(t-{2 * EMBEDDING_DELAY:g})$",
            title="Nearest learned equilibrium",
        )
        ax.legend()

        spectrum_ax = fig.add_subplot(122)
        dde_spectrum = np.asarray(leading_dde_roots[:12])
        ode_spectrum = np.asarray(selected_equilibrium["eigenvalues"])
        spectrum_ax.scatter(
            dde_spectrum.real,
            dde_spectrum.imag,
            s=55,
            label="DDE characteristic roots",
        )
        spectrum_ax.scatter(
            ode_spectrum.real,
            ode_spectrum.imag,
            marker="x",
            s=70,
            label="Learned ODE eigenvalues",
        )
        spectrum_ax.axvline(0.0, color="black", ls="--", lw=0.8)
        spectrum_ax.axhline(0.0, color="black", ls="--", lw=0.8)
        spectrum_ax.set(
            xlabel="Real part",
            ylabel="Imaginary part",
            title="Local spectrum comparison",
        )
        spectrum_ax.legend(fontsize=8)

        plt.tight_layout()
        save_figure(fig, "mg_equilibrium_and_spectrum.png")
        plt.show()
else:
    print("Advanced dynamics skipped. Set RUN_ADVANCED_DYNAMICS = True to run.")


### Experimental two-dimensional local manifolds

The following refactors the separate stable and unstable manifold blocks into one
implementation. It preserves the original truncated-series construction. Treat the
result as an exploratory local approximation; it is not used by the pruning or
forecasting results.


In [ ]:
def compute_local_manifold(
    model,
    equilibrium,
    direction="unstable",
    order=20,
    scale=0.01,
    parameter_range=0.3,
    grid_size=40,
):
    """Construct the original notebook's truncated 2-D manifold approximation."""
    point = equilibrium["z_star"]
    eigenvalues, eigenvectors = eig(equilibrium["jacobian"])
    ordering = np.argsort(eigenvalues.real)
    if direction == "unstable":
        ordering = ordering[::-1]
    elif direction != "stable":
        raise ValueError("direction must be 'stable' or 'unstable'")

    lam1, lam2 = eigenvalues[ordering[:2]]
    vec1, vec2 = eigenvectors[:, ordering[0]], eigenvectors[:, ordering[1]]
    dimension = len(point)

    coefficients = np.zeros((order + 1, order + 1, dimension), dtype=complex)
    coefficients[0, 0] = point
    coefficients[1, 0] = scale * vec1
    coefficients[0, 1] = scale * vec2
    jacobian = numerical_jacobian(model, point)

    def convolution_rhs(m_index, n_index):
        rhs = np.zeros(dimension, dtype=complex)
        for state_index in range(dimension):
            total = 0.0j
            for p in range(m_index):
                r = m_index - 1 - p
                for q in range(n_index):
                    s = n_index - 1 - q
                    if (p > 0 or q > 0) and (r > 0 or s > 0):
                        total += (
                            coefficients[p, q, 0]
                            * coefficients[r, s, state_index]
                        )
            rhs[state_index] = total
        return rhs

    for total_order in range(2, order + 1):
        for m_index in range(1, total_order):
            n_index = total_order - m_index
            if m_index < n_index:
                coefficients[m_index, n_index] = np.conj(
                    coefficients[n_index, m_index]
                )
                continue
            lhs = jacobian - (m_index * lam1 + n_index * lam2) * np.eye(dimension)
            try:
                coefficients[m_index, n_index] = np.linalg.solve(
                    lhs, convolution_rhs(m_index, n_index)
                )
            except np.linalg.LinAlgError:
                coefficients[m_index, n_index] = 0.0

    parameters = np.linspace(-parameter_range, parameter_range, grid_size)
    manifold = np.zeros((grid_size, grid_size, dimension))
    for i, u in enumerate(parameters):
        for j, v in enumerate(parameters):
            sigma1, sigma2 = u + 1j * v, u - 1j * v
            value = np.zeros(dimension, dtype=complex)
            for m_index in range(order + 1):
                for n_index in range(order + 1 - m_index):
                    value += (
                        coefficients[m_index, n_index]
                        * sigma1**m_index
                        * sigma2**n_index
                    )
            manifold[i, j] = value.real

    return {
        "surface": manifold,
        "coefficients": coefficients,
        "eigenvalues": (lam1, lam2),
        "direction": direction,
    }


def evaluate_manifold_series(coefficients, u, v):
    """Evaluate a truncated manifold series at one parameter-space point."""
    sigma1, sigma2 = u + 1j * v, u - 1j * v
    order = coefficients.shape[0] - 1
    value = np.zeros(coefficients.shape[-1], dtype=complex)
    for m_index in range(order + 1):
        for n_index in range(order + 1 - m_index):
            value += (
                coefficients[m_index, n_index]
                * sigma1**m_index
                * sigma2**n_index
            )
    return value.real


def validate_local_manifold(model, manifold_result, duration=20.0):
    """Integrate sample points forward (unstable) or backward (stable)."""
    parameter_points = [
        (0.05, 0.05),
        (-0.05, 0.05),
        (0.05, -0.05),
        (-0.05, -0.05),
        (0.05, 0.0),
        (0.0, 0.05),
        (-0.05, 0.0),
        (0.0, -0.05),
    ]
    direction = manifold_result["direction"]
    end_time = duration if direction == "unstable" else -duration

    def ode_rhs(_, state):
        return model_rhs(model, state)

    trajectories = []
    for u, v in parameter_points:
        initial_state = evaluate_manifold_series(
            manifold_result["coefficients"], u, v
        )
        try:
            solution = solve_ivp(
                ode_rhs,
                (0.0, end_time),
                initial_state,
                max_step=0.05,
                rtol=1e-8,
                atol=1e-10,
            )
            if solution.success:
                trajectories.append(solution.y.T)
        except Exception:
            continue
    return trajectories


if RUN_ADVANCED_DYNAMICS and equilibria:
    local_unstable_manifold = compute_local_manifold(
        model,
        equilibria[0],
        direction="unstable",
        order=40,
        scale=0.1,
        parameter_range=0.7,
    )
    surface = local_unstable_manifold["surface"]
    validation_trajectories = validate_local_manifold(
        model, local_unstable_manifold
    )

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        surface[:, :, 0],
        surface[:, :, 1],
        surface[:, :, 2],
        alpha=0.5,
        cmap="plasma",
        edgecolor="none",
    )
    ax.plot(
        X_delayed[:, 0],
        X_delayed[:, 1],
        X_delayed[:, 2],
        "r--",
        lw=0.4,
        alpha=0.3,
    )
    for trajectory in validation_trajectories:
        ax.plot(
            trajectory[:, 0],
            trajectory[:, 1],
            trajectory[:, 2],
            color="darkgreen",
            lw=0.9,
            alpha=0.7,
        )
    ax.set_title("Experimental local unstable manifold")
    save_figure(fig, "mg_experimental_unstable_manifold.png")
    plt.show()


## 9. Shared pruning utilities

`CoefficientModel` evaluates a dense polynomial regression feature library with a supplied coefficient
matrix. It supports both entry-level masked refitting and feature-level zeroing without
duplicating the rollout and metric code.


In [ ]:
class CoefficientModel:
    """A fitted feature library paired with an explicit coefficient matrix."""

    def __init__(self, feature_library, coefficients):
        self.feature_library = feature_library
        self._coefficients = np.asarray(coefficients).copy()

    def predict(self, X):
        X = np.atleast_2d(X)
        theta = np.asarray(self.feature_library.transform(X))
        return theta @ self._coefficients.T

    def coefficients(self):
        return self._coefficients.copy()


def refit_with_mask(base_model, X, Xdot, mask, ridge=1e-10):
    """Refit every state equation using only coefficient entries allowed by mask."""
    theta = np.asarray(base_model.feature_library.transform(X))
    Xdot = np.asarray(Xdot)
    coefficients = np.zeros_like(mask, dtype=float)

    for equation in range(mask.shape[0]):
        keep = mask[equation]
        if not np.any(keep):
            continue
        design = theta[:, keep]
        lhs = design.T @ design + ridge * np.eye(design.shape[1])
        rhs = design.T @ Xdot[:, equation]
        coefficients[equation, keep] = np.linalg.solve(lhs, rhs)
    return coefficients


def masked_refit_model(base_model, X, Xdot, n_prune, ranking="contribution"):
    coefficients = np.asarray(base_model.coefficients())
    if ranking == "coefficient":
        score = np.abs(coefficients)
    elif ranking == "contribution":
        theta = np.asarray(base_model.feature_library.transform(X))
        feature_rms = np.sqrt(np.mean(theta**2, axis=0))
        score = np.abs(coefficients) * feature_rms[None, :]
    else:
        raise ValueError("ranking must be 'coefficient' or 'contribution'")

    mask = np.ones_like(coefficients, dtype=bool)
    prune_order = np.argsort(score, axis=None)
    mask.ravel()[prune_order[:n_prune]] = False
    refit = refit_with_mask(base_model, X, Xdot, mask)
    return CoefficientModel(base_model.feature_library, refit), mask


def feature_pruned_model(base_model, n_prune):
    """Zero whole library features ranked by coefficient L2 norm; do not refit."""
    coefficients = np.asarray(base_model.coefficients()).copy()
    order = np.argsort(np.linalg.norm(coefficients, axis=0))
    coefficients[:, order[:n_prune]] = 0.0
    return CoefficientModel(base_model.feature_library, coefficients)


def pruning_row(model_candidate, X, times, indices):
    metrics = evaluate_model(model_candidate, X, times, indices)
    coefficients = np.asarray(model_candidate.coefficients())
    return {
        **metrics,
        "n_active_entries": int(np.count_nonzero(coefficients)),
        "n_active_features": int(
            np.count_nonzero(np.linalg.norm(coefficients, axis=0))
        ),
    }


### 9.1 Masked-refit coefficient pruning

Coefficient entries are ranked by their estimated contribution
\(|\xi_{ij}|\,\mathrm{RMS}(\Theta_j)\), removed, and the remaining entries are refit.


In [ ]:
def run_masked_refit_sweep(base_model, X, dt):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    n_entries = np.asarray(base_model.coefficients()).size
    prune_counts = [
        count
        for count in (0, 1, 2, 3, 4, 5, 8, 10, 12, 15, 20, 25, 30, 35,
                      40, 50, 60, 70, 80, 90, 100, 110, 120)
        if count < n_entries
    ]
    rows, models = [], {}
    for count in prune_counts:
        candidate, _ = masked_refit_model(
            base_model, X, Xdot, count, ranking="contribution"
        )
        row = {"n_pruned_entries": count, **pruning_row(
            candidate, X, horizon_times, baseline_test_indices
        )}
        rows.append(row)
        models[count] = candidate
        print(row)
    return pd.DataFrame(rows), models


if RUN_PRUNING_SWEEPS:
    df_masked_refit, masked_refit_models = run_masked_refit_sweep(model, X_delayed, DT)
    df_masked_refit.to_csv(
        OUTPUT_DIR / "mg_masked_refit_pruning_sweep.csv", index=False
    )

    plot_horizon = df_masked_refit["mean_horizon"].clip(upper=HORIZON_CAP)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df_masked_refit["n_pruned_entries"], plot_horizon, "o-", color="darkgreen")
    axes[0].set(xlabel="Pruned coefficient entries", ylabel="Mean horizon", title="Horizon")
    axes[1].plot(
        df_masked_refit["n_pruned_entries"],
        df_masked_refit["mean_nrmse"],
        "s-",
        color="crimson",
    )
    axes[1].set(xlabel="Pruned coefficient entries", ylabel="Mean NRMSE", title="NRMSE")
    axes[2].plot(
        df_masked_refit["n_active_entries"], plot_horizon, "D-", color="steelblue"
    )
    axes[2].invert_xaxis()
    axes[2].set(
        xlabel="Active coefficient entries",
        ylabel="Mean horizon",
        title="Parsimony-horizon tradeoff",
    )
    plt.tight_layout()
    save_figure(fig, "mg_masked_refit_pruning_sweep.png")
    plt.show()
else:
    print("Pruning sweeps skipped. Set RUN_PRUNING_SWEEPS = True to run.")


### 9.2 Feature-level magnitude pruning

Entire library columns are ranked by coefficient-vector norm and zeroed without
refitting. This differs from entry-level masking because a feature is removed from all
state equations at once.


In [ ]:
def run_feature_pruning_sweep(base_model, X):
    n_features = np.asarray(base_model.coefficients()).shape[1]
    prune_counts = [
        count
        for count in (0, 1, 2, 3, 4, 5, 8, 10, 12, 15, 20, 25, 30, 35)
        if count < n_features
    ]
    rows, models = [], {}
    for count in prune_counts:
        candidate = feature_pruned_model(base_model, count)
        row = {"n_pruned_features": count, **pruning_row(
            candidate, X, horizon_times, baseline_test_indices
        )}
        rows.append(row)
        models[count] = candidate
        print(row)
    return pd.DataFrame(rows), models


if RUN_PRUNING_SWEEPS:
    df_feature_pruning, feature_pruned_models = run_feature_pruning_sweep(
        model, X_delayed
    )
    df_feature_pruning.to_csv(
        OUTPUT_DIR / "mg_magnitude_pruning_sweep.csv", index=False
    )

    plot_horizon = df_feature_pruning["mean_horizon"].clip(upper=HORIZON_CAP)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(
        df_feature_pruning["n_pruned_features"],
        plot_horizon,
        "o-",
        color="darkgreen",
    )
    axes[0].set(xlabel="Pruned features", ylabel="Mean horizon", title="Horizon")
    axes[1].plot(
        df_feature_pruning["n_pruned_features"],
        df_feature_pruning["mean_nrmse"],
        "s-",
        color="crimson",
    )
    axes[1].set(xlabel="Pruned features", ylabel="Mean NRMSE", title="NRMSE")
    axes[2].plot(
        df_feature_pruning["n_active_features"],
        plot_horizon,
        "D-",
        color="steelblue",
    )
    axes[2].invert_xaxis()
    axes[2].set(
        xlabel="Active features",
        ylabel="Mean horizon",
        title="Parsimony-horizon tradeoff",
    )
    plt.tight_layout()
    save_figure(fig, "mg_magnitude_pruning_sweep.png")
    plt.show()


## 10. Polynomial and Fourier library comparison

Candidate libraries are fit and evaluated on the same data, rollout starts, bounds,
horizon definition, and NRMSE definition.


In [ ]:
class AblationLibrary:
    """Coordinatewise Fourier features, optionally appended to polynomials."""
    def __init__(self, kind, dimension, degree, periods, harmonics):
        self.kind, self.periods, self.harmonics = kind, periods, harmonics
        self.polynomial = PolynomialLibrary(polynomial_exponents(dimension, degree))

    def transform(self, X):
        X = np.atleast_2d(X)
        if self.kind == "polynomial":
            return self.polynomial.transform(X)
        fourier = [f(2*np.pi*h*X/p) for p in self.periods
                   for h in self.harmonics for f in (np.sin, np.cos)]
        base = self.polynomial.transform(X) if self.kind == "polynomial_fourier" else np.ones((len(X), 1))
        return np.column_stack([base, *fourier])


def run_library_comparison(X, dt):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    amplitude_range = np.ptp(X[:, 0])
    period_sets = {
        "range": (amplitude_range,),
        "half_range": (0.5 * amplitude_range,),
        "double_range": (2.0 * amplitude_range,),
    }
    harmonic_sets = {"k1": (1,), "k1_k2": (1, 2), "k1_k2_k3": (1, 2, 3)}
    rows, models = [], {}

    for kind in ("polynomial", "fourier", "polynomial_fourier"):
        periods_to_try = {"none": (2 * np.pi,)} if kind == "polynomial" else period_sets
        harmonics_to_try = {"none": (1,)} if kind == "polynomial" else harmonic_sets

        for period_label, periods in periods_to_try.items():
            for harmonic_label, harmonics in harmonics_to_try.items():
                tag = (kind, period_label, harmonic_label)
                try:
                    library = AblationLibrary(kind, X.shape[1], 2, periods, harmonics)
                    coefficients, _ = solve_dense_regression(library.transform(X), Xdot)
                    candidate = LibraryModel(coefficients, library)
                    metrics = evaluate_model(
                        candidate, X, horizon_times, baseline_test_indices
                    )
                    coefficients = np.asarray(candidate.coefficients())
                    row = {
                        "library": kind,
                        "period_label": period_label,
                        "harmonics": str(harmonics),
                        **metrics,
                        "n_features": coefficients.shape[1],
                        "n_active_entries": int(np.count_nonzero(coefficients)),
                        "cond_theta": float(np.linalg.cond(library.transform(X))),
                    }
                    rows.append(row)
                    models[tag] = candidate
                    print(row)
                except Exception as exc:
                    print(f"Skipped {tag}: {exc}")

    return pd.DataFrame(rows).sort_values(
        ["stable_fraction", "mean_horizon", "mean_nrmse"],
        ascending=[False, False, True],
    ), models


if RUN_LIBRARY_COMPARISON:
    df_library_comparison, library_models = run_library_comparison(X_delayed, DT)
    df_library_comparison.to_csv(
        OUTPUT_DIR / "mg_library_comparison.csv", index=False
    )
    display(df_library_comparison)
else:
    print("Library comparison skipped. Set RUN_LIBRARY_COMPARISON = True to run.")


## 11. Stepwise quadratic pruning and cubic augmentation

Phase A removes the active quadratic monomial with the smallest coefficient norm and
refits the remaining terms. Phase B starts from the pruned model and greedily adds the
cubic monomial that most reduces the training residual. The figure cell implements the
paper caption for `MG_Prune_Add_Sweep.png`.


In [ ]:
def build_polynomial_exponents(state_dimension, max_degree):
    exponents, degrees = [], []
    for degree in range(max_degree + 1):
        for combination in combinations_with_replacement(range(state_dimension), degree):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combination:
                exponent[index] += 1
            exponents.append(exponent)
            degrees.append(degree)
    return np.asarray(exponents), np.asarray(degrees)


def polynomial_transform(X, exponents):
    X = np.atleast_2d(X)
    return np.prod(X[:, None, :] ** exponents[None, :, :], axis=2)


class PolynomialCoefficientModel:
    def __init__(self, exponents, coefficients):
        self.exponents = np.asarray(exponents)
        self._coefficients = np.asarray(coefficients)

    def predict(self, X):
        return polynomial_transform(X, self.exponents) @ self._coefficients

    def coefficients(self):
        return self._coefficients.T


def fit_active_polynomial(theta, Xdot, active_mask):
    active = np.flatnonzero(active_mask)
    fitted, *_ = np.linalg.lstsq(theta[:, active], Xdot, rcond=None)
    coefficients = np.zeros((theta.shape[1], Xdot.shape[1]))
    coefficients[active] = fitted
    return coefficients


def polynomial_feature_names(exponents):
    names = []
    for exponent in exponents:
        factors = []
        for index, power in enumerate(exponent):
            if power == 1:
                factors.append(f"x{index}")
            elif power > 1:
                factors.append(f"x{index}^{power}")
        names.append("1" if not factors else " ".join(factors))
    return names


def run_prune_add_sweep(X, dt, prune_steps=8, add_steps=8):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    exponents, degrees = build_polynomial_exponents(X.shape[1], max_degree=3)
    names = polynomial_feature_names(exponents)
    theta = polynomial_transform(X, exponents)
    quadratic_or_less = degrees <= 2
    quadratic = degrees == 2
    cubic = degrees == 3

    rows, models = [], {}

    def record(stage, step, mask, coefficients, **action):
        candidate = PolynomialCoefficientModel(exponents, coefficients)
        metrics = evaluate_model(
            candidate, X, horizon_times, baseline_test_indices
        )
        residual = float(np.linalg.norm(theta @ coefficients - Xdot))
        row = {
            "step": step,
            "stage": stage,
            "n_active_features": int(mask.sum()),
            "cond_theta": float(np.linalg.cond(theta[:, mask])),
            "train_residual": residual,
            **metrics,
            **action,
        }
        rows.append(row)
        models[step] = candidate
        print(row)

    active_mask = quadratic_or_less.copy()
    coefficients = fit_active_polynomial(theta, Xdot, active_mask)
    record("baseline_quadratic", 0, active_mask, coefficients)

    for step in range(1, prune_steps + 1):
        norms = np.linalg.norm(coefficients, axis=1)
        candidates = np.where(active_mask & quadratic, norms, np.inf)
        weakest = int(np.argmin(candidates))
        if not np.isfinite(candidates[weakest]):
            break
        active_mask[weakest] = False
        coefficients = fit_active_polynomial(theta, Xdot, active_mask)
        record(
            "prune_quadratic",
            step,
            active_mask,
            coefficients,
            removed_feature=names[weakest],
        )

    remaining_cubic = list(np.flatnonzero(cubic & ~active_mask))
    for addition in range(1, add_steps + 1):
        best_feature, best_residual, best_coefficients = None, np.inf, None
        for feature in remaining_cubic:
            trial_mask = active_mask.copy()
            trial_mask[feature] = True
            trial_coefficients = fit_active_polynomial(theta, Xdot, trial_mask)
            residual = np.linalg.norm(theta @ trial_coefficients - Xdot)
            if residual < best_residual:
                best_feature = feature
                best_residual = residual
                best_coefficients = trial_coefficients

        if best_feature is None:
            break
        active_mask[best_feature] = True
        remaining_cubic.remove(best_feature)
        coefficients = best_coefficients
        record(
            "add_cubic",
            prune_steps + addition,
            active_mask,
            coefficients,
            added_feature=names[best_feature],
        )

    return pd.DataFrame(rows), models


def plot_prune_add_sweep(results):
    """Create the three-panel figure described by the paper caption."""
    baseline = results.iloc[0]
    prune = results[results["stage"] == "prune_quadratic"]
    add = results[results["stage"] == "add_cubic"]
    metrics = [
        ("mean_horizon", "Mean predictive horizon (t.u.)", False),
        ("mean_nrmse", "Mean NRMSE", False),
        ("cond_theta", r"Library condition number $\kappa(\Theta)$", True),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    marker_scale = 260.0

    for ax, (metric, ylabel, log_scale) in zip(axes, metrics):
        ax.axhline(
            baseline[metric],
            color="0.55",
            ls=":",
            lw=1.5,
            label="Dense quadratic baseline",
        )
        ax.scatter(
            baseline["step"],
            baseline[metric],
            s=marker_scale * baseline["stable_fraction"],
            color="0.35",
            edgecolor="black",
            lw=0.6,
            zorder=4,
        )

        ax.plot(
            np.r_[baseline["step"], prune["step"]],
            np.r_[baseline[metric], prune[metric]],
            color="forestgreen",
            lw=2,
        )
        ax.scatter(
            prune["step"],
            prune[metric],
            s=marker_scale * prune["stable_fraction"],
            color="forestgreen",
            edgecolor="black",
            lw=0.5,
            label="Quadratic pruning",
            zorder=4,
        )

        if len(add):
            phase_b_start = prune.iloc[-1] if len(prune) else baseline
            ax.plot(
                np.r_[phase_b_start["step"], add["step"]],
                np.r_[phase_b_start[metric], add[metric]],
                color="rebeccapurple",
                lw=2,
            )
            ax.scatter(
                add["step"],
                add[metric],
                s=marker_scale * add["stable_fraction"],
                color="rebeccapurple",
                edgecolor="black",
                lw=0.5,
                label="Greedy cubic addition",
                zorder=4,
            )

        ax.set(xlabel="Sweep step", ylabel=ylabel)
        ax.set_xticks(results["step"])
        if log_scale:
            ax.set_yscale("log")

    axes[0].set_title("Predictive horizon")
    axes[1].set_title("Rollout error")
    axes[2].set_title("Regression conditioning")
    axes[0].legend(fontsize=8)
    return fig


if RUN_PRUNE_ADD_SWEEP:
    df_prune_add, prune_add_models = run_prune_add_sweep(X_delayed, DT)
    df_prune_add.to_csv(
        OUTPUT_DIR / "mg_prune_quadratic_add_cubic_sweep.csv", index=False
    )
    figure = plot_prune_add_sweep(df_prune_add)
    save_figure(figure, "MG_Prune_Add_Sweep.png", dpi=300)
    plt.show()
else:
    print("Prune/add sweep skipped. Set RUN_PRUNE_ADD_SWEEP = True to run.")


## 12. Measurement-noise robustness

Gaussian noise is added only to the training series. Each noisy model is evaluated
against a clean held-out trajectory, isolating the effect of identification noise from
evaluation noise.


In [ ]:
def run_noise_robustness_sweep(
    clean_series,
    noise_levels=(0.0, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20),
    repeats=5,
    degree=3,
):
    test_start = TRAIN_END_STEPS + TAU_STEPS
    test_end = TEST_END_STEPS
    X_test_clean = build_delay_embedding(
        clean_series,
        test_start,
        test_end,
        delay_steps,
        EMBEDDING_DIMENSION,
    )
    test_times = np.arange(0.0, 40.0 + DT, DT)
    test_indices = evenly_spaced_test_indices(
        len(X_test_clean), len(test_times), n_initial_conditions=12
    )

    rows = []
    for noise_fraction in noise_levels:
        for repeat in range(repeats):
            local_rng = np.random.default_rng(
                int(1_000_000 * noise_fraction) + repeat
            )
            noisy_series = clean_series + local_rng.normal(
                0.0,
                noise_fraction * np.std(clean_series),
                size=clean_series.shape,
            )
            X_train_noisy = build_delay_embedding(
                noisy_series,
                target_start_step,
                target_end_step,
                delay_steps,
                EMBEDDING_DIMENSION,
            )

            try:
                noisy_model = fit_delay_polynomial_model(
                    X_train_noisy, DT, degree=degree
                )
                metrics = evaluate_model(
                    noisy_model,
                    X_test_clean,
                    test_times,
                    test_indices,
                    bounds=(0.0, 3.0),
                )
            except Exception as exc:
                metrics = {
                    "stable_rollouts": 0,
                    "stable_fraction": 0.0,
                    "mean_horizon": np.nan,
                    "mean_nrmse": np.nan,
                }
                print(f"Noise={noise_fraction}, repeat={repeat} failed: {exc}")

            rows.append(
                {
                    "noise_fraction": noise_fraction,
                    "repeat": repeat,
                    **metrics,
                }
            )
    return pd.DataFrame(rows)


if RUN_NOISE_SWEEP:
    df_noise = run_noise_robustness_sweep(data)
    df_noise.to_csv(OUTPUT_DIR / "mg_noise_results.csv", index=False)
    noise_summary = (
        df_noise.groupby("noise_fraction")
        .agg(
            mean_horizon=("mean_horizon", "mean"),
            std_horizon=("mean_horizon", "std"),
            mean_nrmse=("mean_nrmse", "mean"),
            std_nrmse=("mean_nrmse", "std"),
            mean_stable_fraction=("stable_fraction", "mean"),
        )
        .reset_index()
    )
    display(noise_summary)

    x_percent = 100 * noise_summary["noise_fraction"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].errorbar(
        x_percent,
        noise_summary["mean_horizon"],
        yerr=noise_summary["std_horizon"],
        marker="o",
        color="steelblue",
    )
    axes[0].set(
        xlabel="Noise level (% of signal std)",
        ylabel="Mean predictive horizon (t.u.)",
        title="Horizon vs. measurement noise",
    )
    axes[1].errorbar(
        x_percent,
        noise_summary["mean_nrmse"],
        yerr=noise_summary["std_nrmse"],
        marker="o",
        color="steelblue",
    )
    axes[1].set(
        xlabel="Noise level (% of signal std)",
        ylabel="Mean NRMSE",
        title="NRMSE vs. measurement noise",
    )
    plt.tight_layout()
    save_figure(fig, "mg_noise_robustness.png")
    plt.show()
else:
    print("Noise sweep skipped. Set RUN_NOISE_SWEEP = True to run.")


## 13. Surrogate-only periodic-orbit candidate for CAP

The baseline cubic learned orbit is reproduced first. A degree-5 model of the
same polynomial/ridge class is then fitted to finer-resolution samples from the
same explicit-Euler generator. Both shooting and Fourier refinement below use
only the learned ODE vector field. The true Mackey–Glass equation is not queried
between model fitting and the CAP call.


### 13.1 Learned-ODE shooting and Fourier utilities


In [ ]:
"""Periodic-orbit utilities for the learned Mackey--Glass ODE surrogate."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.spatial import cKDTree


def scan_periodic_recurrences(
    trajectory,
    dt,
    min_period=20.0,
    max_period=250.0,
    search_stride=5,
    neighbors=80,
    max_candidates=30,
):
    """Find and rank close returns of a learned trajectory with a KD tree."""
    trajectory = np.asarray(trajectory, dtype=float)
    sampled_indices = np.arange(0, len(trajectory), search_stride)
    sampled = trajectory[sampled_indices]
    scale = np.std(sampled, axis=0)
    scale[scale < 1e-10] = 1.0
    standardized = (sampled - np.mean(sampled, axis=0)) / scale
    tree = cKDTree(standardized)
    query_count = min(neighbors + 1, len(standardized))
    distances, neighbor_indices = tree.query(
        standardized, k=query_count, workers=-1
    )

    minimum_lag = int(round(min_period / dt))
    maximum_lag = int(round(max_period / dt))
    raw = []
    for local_start, (row_distances, row_neighbors) in enumerate(
        zip(distances, neighbor_indices)
    ):
        start = int(sampled_indices[local_start])
        for distance, local_end in zip(row_distances[1:], row_neighbors[1:]):
            end = int(sampled_indices[int(local_end)])
            if end <= start:
                continue
            lag = end - start
            if minimum_lag <= lag <= maximum_lag:
                raw.append((float(distance), start, end))

    raw.sort(key=lambda item: item[0])
    rows = []
    start_tolerance = max(1, int(round(2.0 / dt)))
    for standardized_distance, start, end in raw:
        period = (end - start) * dt
        duplicate = any(
            abs(start - old["start_index"]) <= start_tolerance
            and abs(period - old["period"]) <= 2.0
            for old in rows
        )
        if duplicate:
            continue

        segment = trajectory[start : end + 1]
        return_error = float(np.linalg.norm(trajectory[end] - trajectory[start]))
        diameter = float(np.max(np.linalg.norm(segment - segment[0], axis=1)))
        rows.append(
            {
                "start_index": start,
                "end_index": end,
                "period": float(period),
                "standardized_distance": standardized_distance,
                "return_error": return_error,
                "segment_diameter": diameter,
                "relative_return_error": float(
                    return_error / (diameter + 1e-12)
                ),
            }
        )
        if len(rows) >= max_candidates:
            break

    columns = [
        "start_index",
        "end_index",
        "period",
        "standardized_distance",
        "return_error",
        "segment_diameter",
        "relative_return_error",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows).sort_values(
        ["relative_return_error", "standardized_distance"],
        ignore_index=True,
    )


def adaptive_endpoint(
    initial_state,
    period,
    rhs_function,
    rtol=1e-9,
    atol=1e-11,
    max_step=0.1,
    max_abs=10.0,
):
    """High-accuracy endpoint of the learned autonomous ODE."""
    initial_state = np.asarray(initial_state, dtype=float)
    solution = solve_ivp(
        lambda _time, state: rhs_function(state),
        (0.0, float(period)),
        initial_state,
        method="DOP853",
        t_eval=[float(period)],
        rtol=rtol,
        atol=atol,
        max_step=max_step,
    )
    if (
        not solution.success
        or solution.y.shape[1] == 0
        or not np.all(np.isfinite(solution.y[:, -1]))
        or np.max(np.abs(solution.y[:, -1])) > max_abs
    ):
        return np.full_like(initial_state, np.nan)
    return solution.y[:, -1]


def periodic_closure_defect(state, endpoint):
    """Normalized full-state periodicity defect."""
    state = np.asarray(state, dtype=float)
    endpoint = np.asarray(endpoint, dtype=float)
    scale = max(np.linalg.norm(state - np.mean(state)), 1e-12)
    return float(np.linalg.norm(endpoint - state) / scale)


@dataclass
class PeriodicOrbitRefinement:
    state: np.ndarray
    period: float
    endpoint: np.ndarray
    optimizer_success: bool
    converged: bool
    nfev: int
    cost: float
    relative_closure_defect: float
    time_phase_defect: float
    relative_state_correction: float
    speed_norm: float
    message: str


def refine_periodic_orbit(
    reference_state,
    period_guess,
    endpoint_function,
    rhs_function,
    max_nfev=80,
    defect_tolerance=1e-8,
    period_window=(0.7, 1.3),
    state_bounds=(0.0, 2.0),
):
    """Single-shoot a learned periodic orbit with one time-phase condition."""
    reference_state = np.asarray(reference_state, dtype=float)
    n_state = len(reference_state)
    time_tangent = np.asarray(rhs_function(reference_state), dtype=float)
    state_scale = max(
        np.linalg.norm(reference_state - np.mean(reference_state)), 1e-8
    )
    time_scale = max(state_scale * np.linalg.norm(time_tangent), 1e-10)
    initial = np.concatenate([reference_state, [np.log(period_guess)]])
    lower = np.full_like(initial, -np.inf)
    upper = np.full_like(initial, np.inf)
    lower[:n_state] = state_bounds[0]
    upper[:n_state] = state_bounds[1]
    lower[-1] = np.log(period_window[0] * period_guess)
    upper[-1] = np.log(period_window[1] * period_guess)

    def unpack(parameters):
        return parameters[:-1], float(np.exp(parameters[-1]))

    def residual(parameters):
        state, period = unpack(parameters)
        endpoint = endpoint_function(state, period)
        if not np.all(np.isfinite(endpoint)):
            return np.full(n_state + 1, 1e6)
        closure = (endpoint - state) / state_scale
        displacement = state - reference_state
        time_phase = float(np.dot(displacement, time_tangent) / time_scale)
        return np.concatenate([closure, [time_phase]])

    result = least_squares(
        residual,
        initial,
        bounds=(lower, upper),
        max_nfev=max_nfev,
        xtol=1e-10,
        ftol=1e-10,
        gtol=1e-10,
        x_scale="jac",
        verbose=0,
    )
    state, period = unpack(result.x)
    endpoint = endpoint_function(state, period)
    relative_defect = periodic_closure_defect(state, endpoint)
    displacement = state - reference_state
    time_phase = float(np.dot(displacement, time_tangent) / time_scale)
    state_correction = float(np.linalg.norm(displacement) / state_scale)
    speed_norm = float(np.linalg.norm(rhs_function(state)))
    converged = bool(
        result.success
        and relative_defect < defect_tolerance
        and speed_norm > 1e-6
        and state_correction < 1.0
    )
    return PeriodicOrbitRefinement(
        state=state,
        period=period,
        endpoint=endpoint,
        optimizer_success=bool(result.success),
        converged=converged,
        nfev=int(result.nfev),
        cost=float(result.cost),
        relative_closure_defect=relative_defect,
        time_phase_defect=time_phase,
        relative_state_correction=state_correction,
        speed_norm=speed_norm,
        message=str(result.message),
    )


def sample_learned_periodic_orbit(
    state,
    period,
    rhs_function,
    n_samples=1024,
    rtol=1e-11,
    atol=1e-13,
    max_step=0.05,
):
    """Sample component x(t) over one learned-model period."""
    times = np.linspace(0.0, float(period), n_samples, endpoint=False)
    solution = solve_ivp(
        lambda _time, current: rhs_function(current),
        (0.0, float(period)),
        np.asarray(state, dtype=float),
        method="DOP853",
        t_eval=times,
        rtol=rtol,
        atol=atol,
        max_step=max_step,
    )
    if not solution.success or solution.y.shape[1] != len(times):
        raise RuntimeError("Could not sample the learned periodic orbit.")
    return times, solution.y[0]


def fourier_phase_design(phases, harmonics):
    """Vectorized real Fourier design on phases measured in cycles."""
    phases = np.atleast_1d(np.asarray(phases, dtype=float))
    design = np.empty((len(phases), 2 * harmonics + 1), dtype=float)
    design[:, 0] = 1.0
    harmonic = np.arange(1, harmonics + 1, dtype=float)
    angles = 2.0 * np.pi * phases[:, None] * harmonic[None, :]
    design[:, 1::2] = np.cos(angles)
    design[:, 2::2] = np.sin(angles)
    return design


def fourier_phase_derivative_design(phases, harmonics):
    """Derivative of the Fourier design with respect to cycle phase."""
    phases = np.atleast_1d(np.asarray(phases, dtype=float))
    design = np.zeros((len(phases), 2 * harmonics + 1), dtype=float)
    harmonic = np.arange(1, harmonics + 1, dtype=float)
    angles = 2.0 * np.pi * phases[:, None] * harmonic[None, :]
    angular_frequency = 2.0 * np.pi * harmonic[None, :]
    design[:, 1::2] = -angular_frequency * np.sin(angles)
    design[:, 2::2] = angular_frequency * np.cos(angles)
    return design


def fourier_design(times, period, harmonics):
    """Real Fourier design matrix [1, cos(k wt), sin(k wt)]."""
    return fourier_phase_design(
        np.asarray(times, dtype=float) / float(period), harmonics
    )


def fit_periodic_history(times, values, period, harmonics=12):
    """Least-squares Fourier representation of a periodic scalar waveform."""
    design = fourier_design(times, period, harmonics)
    return np.linalg.lstsq(design, np.asarray(values, dtype=float), rcond=None)[0]


def evaluate_periodic_history(times, coefficients, period):
    coefficients = np.asarray(coefficients, dtype=float)
    harmonics = (len(coefficients) - 1) // 2
    return fourier_design(times, period, harmonics) @ coefficients


def evaluate_periodic_history_derivative(times, coefficients, period):
    coefficients = np.asarray(coefficients, dtype=float)
    times = np.asarray(times, dtype=float)
    harmonics = (len(coefficients) - 1) // 2
    phase_derivative = fourier_phase_derivative_design(
        times / float(period), harmonics
    )
    return phase_derivative @ coefficients / float(period)




### 13.2 Degree-5 surrogate orbit and direct CAP export


In [ ]:
def generate_learned_attractor_trajectory(
    model,
    initial_state,
    dt,
    burn_time=500.0,
    sample_time=3000.0,
):
    times = np.arange(0.0, burn_time + sample_time + dt, dt)
    trajectory = safe_rk4_rollout(model, initial_state, times)
    if trajectory is None:
        raise RuntimeError("Learned-model orbit-search rollout diverged.")
    burn_steps = int(round(burn_time / dt))
    return trajectory[burn_steps:]


ORBIT_PROPOSAL_TRAINING_WINDOWS = [
    (117.0, 417.0),
    (700.0, 1000.0),
    (1100.0, 1400.0),
    (2700.0, 3000.0),
    (3100.0, 3400.0),
]
ORBIT_PROPOSAL_RIDGE_ALPHA = 5e-5
ORBIT_PROPOSAL_PERIOD_BAND = (15.0, 80.0)


def build_orbit_polynomial_exponents(state_dimension, max_degree):
    """Build one exponent array without relying on later notebook helpers."""
    exponents = [np.zeros(state_dimension, dtype=int)]
    for degree in range(1, max_degree + 1):
        for combo in combinations_with_replacement(range(state_dimension), degree):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combo:
                exponent[index] += 1
            exponents.append(exponent)
    return np.asarray(exponents)


def fit_orbit_proposal_model(training_windows):
    """Fit the same polynomial model class on several disjoint orbit windows."""
    powers = build_orbit_polynomial_exponents(
        EMBEDDING_DIMENSION, BASELINE_POLY_DEGREE
    )
    library = PolynomialLibrary(powers)
    theta_blocks = []
    derivative_blocks = []
    delay_steps = int(round(EMBEDDING_DELAY / DT))
    for start_time, end_time in training_windows:
        embedded = build_delay_embedding(
            data,
            int(round(start_time / DT)),
            int(round(end_time / DT)),
            delay_steps,
            EMBEDDING_DIMENSION,
        )
        theta_blocks.append(library.transform(embedded))
        derivative_blocks.append(
            np.gradient(embedded, DT, axis=0, edge_order=2)
        )

    theta = np.vstack(theta_blocks)
    derivatives = np.vstack(derivative_blocks)
    feature_degree = powers.sum(axis=1)
    penalty = np.maximum(feature_degree, 1) ** RIDGE_DEGREE_WEIGHT
    penalty[0] = 0.0
    coefficients, _ = solve_dense_regression(
        theta, derivatives, regularization=ORBIT_PROPOSAL_RIDGE_ALPHA,
        normalize=True, weights=penalty,
    )
    return PolynomialModel(coefficients, powers)




def fit_original_basis_model(series, degree=5, alpha=1e-7):
    powers = build_orbit_polynomial_exponents(
        EMBEDDING_DIMENSION, degree
    )
    library = PolynomialLibrary(powers)
    delay_steps = int(round(EMBEDDING_DELAY / DT))
    theta_blocks = []
    derivative_blocks = []
    for start, end in ORBIT_PROPOSAL_TRAINING_WINDOWS:
        embedded = build_delay_embedding(
            series,
            int(round(start / DT)),
            int(round(end / DT)),
            delay_steps,
            EMBEDDING_DIMENSION,
        )
        theta_blocks.append(library.transform(embedded))
        derivative_blocks.append(
            np.gradient(embedded, DT, axis=0, edge_order=2)
        )
    theta = np.vstack(theta_blocks)
    derivatives = np.vstack(derivative_blocks)
    feature_degree = powers.sum(axis=1)
    penalty = np.maximum(feature_degree, 1) ** RIDGE_DEGREE_WEIGHT
    penalty[0] = 0.0
    coefficients, _ = solve_dense_regression(
        theta, derivatives, regularization=alpha, normalize=True, weights=penalty,
    )
    return PolynomialModel(coefficients, powers)

def refine_on_surrogate_delay_manifold(
    learned_model, seed_coefficients, period_guess, harmonics=40, node_count=384
):
    coefficient_count = 2 * harmonics + 1
    seed = np.zeros(coefficient_count)
    seed[: min(coefficient_count, len(seed_coefficients))] = seed_coefficients[
        :coefficient_count
    ]
    phases = (np.arange(node_count) + 0.317) / node_count
    seed_waveform = fourier_phase_design(phases, harmonics) @ seed
    seed_tangent = fourier_phase_derivative_design(phases, harmonics) @ seed / period_guess
    derivative_scale = max(float(np.sqrt(np.mean(seed_tangent**2))), 1e-4)
    phase_scale = max(
        float(np.linalg.norm(seed_tangent) * np.std(seed_waveform)), 1e-8
    )
    mean_scale = max(float(np.std(seed_waveform)), 1e-8)
    seed_energy = float(np.mean((seed_waveform - np.mean(seed_waveform)) ** 2))
    energy_scale = max(seed_energy, 1e-8)
    initial = np.r_[seed, np.log(period_guess)]
    lower = np.r_[np.full(coefficient_count, -np.inf), np.log(0.95 * period_guess)]
    upper = np.r_[np.full(coefficient_count, np.inf), np.log(1.05 * period_guess)]

    def residual(parameters):
        coefficients = parameters[:-1]
        period = float(np.exp(parameters[-1]))
        coordinates = []
        derivatives = []
        for coordinate in range(EMBEDDING_DIMENSION):
            shifted_phase = (
                phases - coordinate * EMBEDDING_DELAY / period
            )
            coordinates.append(fourier_phase_design(shifted_phase, harmonics) @ coefficients)
            derivatives.append(
                fourier_phase_derivative_design(shifted_phase, harmonics) @ coefficients / period
            )
        states = np.column_stack(coordinates)
        state_derivatives = np.column_stack(derivatives)
        learned_equation = (
            state_derivatives - learned_model.predict(states)
        ).ravel() / derivative_scale
        waveform = states[:, 0]
        phase_condition = np.dot(
            waveform - seed_waveform, seed_tangent
        ) / phase_scale
        mean_condition = (np.mean(waveform) - np.mean(seed_waveform)) / mean_scale
        energy = np.mean((waveform - np.mean(waveform)) ** 2)
        energy_condition = (energy - seed_energy) / energy_scale
        return np.r_[
            learned_equation, phase_condition, mean_condition, energy_condition
        ]

    result = least_squares(
        residual,
        initial,
        bounds=(lower, upper),
        max_nfev=50,
        xtol=2e-12,
        ftol=2e-12,
        gtol=2e-12,
        x_scale="jac",
        verbose=1,
    )
    return result, result.x[:-1], float(np.exp(result.x[-1])), residual(result.x)


# Reproduce the current notebook's original cubic learned orbit as a continuation
# seed. This stage is unchanged and uses only the learned ODE.
if RUN_PERIODIC_ORBIT_SEARCH:
    baseline_orbit_model = fit_orbit_proposal_model(ORBIT_PROPOSAL_TRAINING_WINDOWS)
    baseline_attractor = generate_learned_attractor_trajectory(
        baseline_orbit_model, X_test[1000], DT, burn_time=500.0, sample_time=3000.0
    )
    baseline_recurrences = scan_periodic_recurrences(
        baseline_attractor,
        DT,
        min_period=ORBIT_PROPOSAL_PERIOD_BAND[0],
        max_period=ORBIT_PROPOSAL_PERIOD_BAND[1],
        search_stride=5,
        neighbors=240,
        max_candidates=80,
    )
    if baseline_recurrences.empty:
        raise RuntimeError("No recurrence seed was found in the cubic surrogate.")
    
    baseline_rhs = lambda state: model_rhs(baseline_orbit_model, state)
    baseline_endpoint = lambda state, period: adaptive_endpoint(
        state,
        period,
        baseline_rhs,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.1,
        max_abs=3.0,
    )
    baseline_seed = baseline_recurrences.iloc[0]
    baseline_orbit = refine_periodic_orbit(
        baseline_attractor[int(baseline_seed["start_index"])],
        float(baseline_seed["period"]),
        baseline_endpoint,
        baseline_rhs,
        max_nfev=80,
        defect_tolerance=1e-8,
        state_bounds=STATE_BOUNDS,
    )
    if not baseline_orbit.converged:
        raise RuntimeError(f"Cubic learned-orbit shooting failed: {baseline_orbit}")
    
    # Only the numerical resolution of the existing explicit-Euler data generator is
    # changed. Observations supplied to the surrogate remain on DT=0.1.
    SURROGATE_POLY_DEGREE = 5
    SURROGATE_RIDGE_ALPHA = 1e-7
    EULER_INTERNAL_DT = 0.001
    EULER_DOWNSAMPLE = int(round(DT / EULER_INTERNAL_DT))
    if not np.isclose(EULER_DOWNSAMPLE * EULER_INTERNAL_DT, DT):
        raise ValueError("EULER_INTERNAL_DT must divide DT exactly.")
    
    _, fine_euler_full = generate_mackey_glass_euler(
        dt=EULER_INTERNAL_DT, t_max=T_MAX
    )
    surrogate_series = fine_euler_full[::EULER_DOWNSAMPLE]
    if len(surrogate_series) != len(data):
        raise RuntimeError("Fine-Euler downsampling did not reproduce the DT grid.")
    
    surrogate_model = fit_original_basis_model(
        surrogate_series,
        degree=SURROGATE_POLY_DEGREE,
        alpha=SURROGATE_RIDGE_ALPHA,
    )
    surrogate_rhs = lambda state: model_rhs(surrogate_model, state)
    surrogate_endpoint = lambda state, period: adaptive_endpoint(
        state,
        period,
        surrogate_rhs,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.1,
        max_abs=3.0,
    )
    surrogate_free_orbit = refine_periodic_orbit(
        baseline_orbit.state,
        baseline_orbit.period,
        surrogate_endpoint,
        surrogate_rhs,
        max_nfev=100,
        defect_tolerance=1e-8,
        period_window=(0.9, 1.1),
        state_bounds=STATE_BOUNDS,
    )
    if not surrogate_free_orbit.converged:
        raise RuntimeError(
            f"Degree-5 learned-orbit continuation failed: {surrogate_free_orbit}"
        )
    
    orbit_times, orbit_signal = sample_learned_periodic_orbit(
        surrogate_free_orbit.state,
        surrogate_free_orbit.period,
        surrogate_rhs,
        n_samples=4096,
        max_step=0.03,
    )
    free_orbit_coefficients = fit_periodic_history(
        orbit_times,
        orbit_signal,
        surrogate_free_orbit.period,
        harmonics=40,
    )
    (
        surrogate_refinement,
        surrogate_candidate_coefficients,
        surrogate_candidate_period,
        surrogate_residual,
    ) = refine_on_surrogate_delay_manifold(
        surrogate_model,
        free_orbit_coefficients,
        surrogate_free_orbit.period,
        harmonics=40,
        node_count=384,
    )
    surrogate_collocation_rms = float(
        np.sqrt(np.mean(surrogate_residual[:-3] ** 2))
    )
    
    # Pad to the CAP truncation without changing any retained Fourier mode.
    cap_candidate_coefficients = np.zeros(401)
    cap_candidate_coefficients[: len(surrogate_candidate_coefficients)] = (
        surrogate_candidate_coefficients
    )
    cap_candidate_path = OUTPUT_DIR / "MG_zero_correction_candidate.mat"
    savemat(
        cap_candidate_path,
        {
            "candidate_real_fourier": cap_candidate_coefficients,
            "period_candidate_physical": surrogate_candidate_period,
        },
    )
    
    surrogate_audit = pd.DataFrame(
        [
            {
                "polynomial_degree": SURROGATE_POLY_DEGREE,
                "ridge_alpha": SURROGATE_RIDGE_ALPHA,
                "euler_internal_dt": EULER_INTERNAL_DT,
                "observation_dt": DT,
                "baseline_cubic_period": baseline_orbit.period,
                "degree5_free_period": surrogate_free_orbit.period,
                "degree5_free_closure": surrogate_free_orbit.relative_closure_defect,
                "exported_period": surrogate_candidate_period,
                "surrogate_fourier_nfev": int(surrogate_refinement.nfev),
                "surrogate_collocation_rms": surrogate_collocation_rms,
                "true_dde_pre_cap_calls": 0,
            }
        ]
    )
    surrogate_audit.to_csv(
        OUTPUT_DIR / "surrogate_only_transfer_audit.csv", index=False
    )
    display(surrogate_audit)
    print(f"Saved raw surrogate CAP candidate to: {cap_candidate_path.resolve()}")
    
    phase = np.linspace(0.0, 1.0, 1500)
    free_waveform = fourier_phase_design(phase, 40) @ free_orbit_coefficients
    exported_waveform = (
        fourier_phase_design(phase, 40) @ surrogate_candidate_coefficients
    )
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(
        phase * surrogate_free_orbit.period,
        free_waveform,
        "--",
        lw=1,
        label="degree-5 shooting orbit",
    )
    axes[0].plot(
        phase * surrogate_candidate_period,
        exported_waveform,
        lw=1.4,
        label="surrogate-only Fourier refinement",
    )
    axes[0].set(xlabel="Time over one period", ylabel="$x(t)$")
    axes[0].legend()
    
    delay_phase = EMBEDDING_DELAY / surrogate_candidate_period
    delayed_waveform = fourier_phase_design(phase - delay_phase, 40) @ (
        surrogate_candidate_coefficients
    )
    axes[1].plot(exported_waveform, delayed_waveform, lw=1.2)
    axes[1].set(
        xlabel="$x(t)$",
        ylabel=f"$x(t-{EMBEDDING_DELAY:g})$",
        title="Exported learned-ODE orbit",
    )
    figure.tight_layout()
    save_figure(figure, "mg_surrogate_only_cap_candidate.png", dpi=250)
    plt.show()
else:
    print("Surrogate-only orbit construction skipped. Set RUN_PERIODIC_ORBIT_SEARCH = True to run.")


## 14. Fixed-parameter CAP with zero Newton corrections

The exported surrogate waveform is validated at fixed feedback 0.2, damping 0.1, delay 17, and exponent 10. Frequency is the unknown scalar. The new driver verifies pole exclusion on the full radius-0.0006 ball and bounds all omitted Fourier modes. It supersedes the former free-damping driver.

The focused notebook `mg_periodic_orbit_certification.ipynb` contains an executed certificate view, the mathematical formulation, and verification checks.


In [ ]:
import json, os, sys, subprocess
PROOF_DIR = REPO_ROOT / "CAP/Mackey-Glass_PO"
PROOF_OUTPUT = PROOF_DIR / "outputs_fixed_parameter"
RUN_INTERVAL_PROOF = os.environ.get("MG_RUN_CAP_FROM_NOTEBOOK", "0") == "1"
if RUN_INTERVAL_PROOF:
    subprocess.run([sys.executable, str(PROOF_DIR / "fixed_parameter_candidate.py"),
                    "--newton-steps", "0"], check=True)
    matlab = os.environ.get("MATLAB_BINARY", "/Applications/MATLAB_R2024b.app/bin/matlab")
    intlab = os.environ.get("INTLAB_DIR", "/Users/aaronloh/Documents/MATLAB/Intlab_V13")
    quote = lambda p: str(p).replace("'", "''")
    command = (f"addpath('{quote(intlab)}'); startintlab; "
               f"run('{quote(PROOF_DIR / 'proof_MG_fixed_parameter.m')}');")
    result = subprocess.run([matlab, "-batch", command,
                             "-logfile", str(PROOF_OUTPUT / "cap.log")],
                            capture_output=True, text=True, timeout=600)
    print(result.stdout[-6000:])
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Fixed-parameter interval proof failed.")
else:
    print("Using the executed MATLAB/INTLAB certificate; no proof rerun in this notebook session.")
certificate = json.loads((PROOF_OUTPUT / "mg_fixed_parameter_certificate.json").read_text())
assert certificate["proof_success"] and certificate["fixed_parameters"]
assert certificate["complete_infinite_tail"]
assert certificate["true_system_newton_steps"] == 0
assert certificate["radii_polynomial_upper"] < 0
assert certificate["contraction_upper"] < 1
assert certificate["pole_neighborhood_factor"] < 1
assert certificate["nonconstant_coefficient_lower"] > 0
print(json.dumps(certificate, indent=2))
